# Cboe 免费期权样本：来源字段预检

只取两个官方公开 ZIP 样本并验原始字节、字段/时间格式与 NAS 回读。样本不证明 SOXX 历史覆盖、25 Delta 插值或策略收益。

In [1]:
import io,os,json,hashlib,subprocess,datetime,zipfile,tarfile,tempfile,shlex
from pathlib import Path
from src.data.nas_market_data import _settings,nas_api_call
root=Path.cwd();card=root/'research/soxx_option_vendor_sample_preflight_20261023';raw=root/'data/snapshots/soxx_option_vendor_sample_preflight_20261023'
plan=json.loads((card/'plan.json').read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and len(plan['exact_GETs'])==2 and not (card/'sample_result.json').exists()
state={'status':'started','GET_attempts':0,'model_fits':0,'native_accounts':0,'SOXX_market_outcome_reads':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
receipts=[];profiles=[]
for spec,stem in zip(plan['exact_GETs'],['option_quotes','option_eod']):
    headers=raw/(stem+'.headers');body=raw/(stem+'.zip')
    start=datetime.datetime.now(datetime.timezone.utc).isoformat()
    cmd=['curl','--http1.1','--noproxy','*','--connect-timeout','10','--max-time','45','--max-filesize','10000000','--fail-with-body','-sS','-L','-D',str(headers),'-o',str(body),'-A','SOXL Qlib research source audit research@atrl.me','-w','%{http_code} %{url_effective}',spec['url']]
    run=subprocess.run(cmd,text=True,capture_output=True,timeout=55)
    end=datetime.datetime.now(datetime.timezone.utc).isoformat()
    state['GET_attempts']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
    code,url=(run.stdout.split(' ',1)+[''])[:2] if run.stdout else ('','')
    receipt={'product':spec['product'],'requested_url':spec['url'],'requested_at_utc':start,'received_at_utc':end,'curl_exit_code':run.returncode,'http_status':code,'final_url':url,'raw_zip_sha256':sha(body) if body.exists() else None,'raw_zip_bytes':body.stat().st_size if body.exists() else None,'headers_sha256':sha(headers) if headers.exists() else None,'stderr':run.stderr[:500]}
    receipts.append(receipt)
    if run.returncode!=0 or code!='200' or not body.exists() or body.stat().st_size>10000000 or not zipfile.is_zipfile(body):break
    with zipfile.ZipFile(body) as z:
        members=[x for x in z.infolist() if not x.is_dir()]
        examples=[]
        for member in members[:20]:
            entry={'name':member.filename,'bytes':member.file_size}
            if member.file_size<=50000000 and member.filename.lower().endswith(('.csv','.txt')):
                with z.open(member) as src:
                    lines=io.TextIOWrapper(src,encoding='utf-8-sig',errors='replace')
                    entry['first_lines']=[line.rstrip('\r\n')[:1000] for _,line in zip(range(3),lines)]
            examples.append(entry)
        profiles.append({'product':spec['product'],'member_count':len(members),'sample_entries':examples,'all_entry_names':[x.filename for x in members[:100]],'ZIP_uncompressed_bytes':sum(x.file_size for x in members)})
(card/'sample_receipts.json').write_text(json.dumps(receipts,indent=2,ensure_ascii=False)+'\n')
(card/'sample_profiles.json').write_text(json.dumps(profiles,indent=2,ensure_ascii=False)+'\n')
manifest={'plan_sha256':sha(card/'plan.json'),'files':[{'name':x.name,'sha256':sha(x),'bytes':x.stat().st_size} for x in sorted(raw.iterdir()) if x.is_file()],'receipts_sha256':sha(card/'sample_receipts.json'),'profiles_sha256':sha(card/'sample_profiles.json')}
(card/'sample_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='soxx-option-vendor-source-') as td:
    archive=Path(td)/'samples.tar.gz'
    with tarfile.open(archive,'w:gz') as t:
        for f in sorted(raw.iterdir()):
            if f.is_file():t.add(f,arcname=f.name)
        t.add(card/'sample_manifest.json',arcname='sample_manifest.json')
    digest=sha(archive)
    artifact=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
    assert artifact['sha256']==digest and artifact['size_bytes']==archive.stat().st_size
    restored=Path(td)/'restored.tar.gz'
    with restored.open('wb') as out:
        run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(artifact['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=120)
    assert run.returncode==0 and sha(restored)==digest
    with tarfile.open(restored,'r:gz') as t:
        names=[x.name for x in t.getmembers() if x.isfile()]
    assert set(names)==set(x['name'] for x in manifest['files'])|{'sample_manifest.json'}
status='two_public_Cboe_ZIP_samples_received_schema_only' if len(receipts)==2 and all(x['curl_exit_code']==0 and x['http_status']=='200' for x in receipts) else 'bounded_sample_partial_or_failed'
result={'status':status,'plan_sha256':manifest['plan_sha256'],'GET_attempts':state['GET_attempts'],'receipts':receipts,'profiles':profiles,'raw_NAS_archive_sha256':digest,'NAS_remote_and_scoped_restore_verified':True,'official_historical_start':'2012-01','SOXX_specific_historical_coverage_proven':False,'2011_Stage1_training_coverage_proven':False,'25_Delta_constant_tenor_feature_qualified':False,'SOXX_market_outcome_reads':0,'model_fits':0,'native_accounts':0,'purchases':0}
(card/'sample_result.json').write_text(json.dumps(result,indent=2,ensure_ascii=False)+'\n')
state.update(status='completed' if status.startswith('two_public') else 'partial_or_failed',NAS_archive_sha256=digest);(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
print({'status':status,'GET_attempts':state['GET_attempts'],'products':[(x['product'],x['http_status'],x['raw_zip_bytes']) for x in receipts],'members':[(x['product'],x['member_count']) for x in profiles],'NAS':digest})

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


KeyError: 'nas_path'

# SOXX 期权偏斜候选：Cboe 免费样本来源预检

按冻结预算只下载 Cboe 官方 [Option Quotes](https://datashop.cboe.com/option-quote-intervals) 与 [Option EOD Summary](https://datashop.cboe.com/option-eod-summary) 两份公开样本，共 2 次 GET、0 购买。原始 ZIP 分别为 2,628,937 和 2,952,557 字节，响应头、ZIP 原字节及 SHA-256 收据见 `sample_receipts.json` 和 `sample_manifest.json`。首轮 NAS 上传通过，但程序误以为回执含 `nas_path` 而未做回读；按冻结机械修复从仓库 NAS 归档根与哈希推导路径，重新归档并限定恢复成功，归档 SHA-256 为 `ba88ddde28a5878107c6ef48d9a76f0149270b369ed80ba09e9ce4466424cf8f`。没有重复网页请求。

Option Quotes 样本含五个内层 CSV 变体；带 Calcs 的表头有报价时间、bid/ask、隐含波动率和 Delta，样本首行标的是 AAPL 或 ^SPX。EOD 样本有 15:45 报价、IV/Delta 和日终字段，首行是 SPY。样本仅证明字段格式，不证明 SOXX 个股期权链覆盖。两份产品页目前均声明历史起点为 **2012 年 1 月**，不能据此前 FAQ 的宽泛“2010 年”说法填补冻结 Stage1 的 2011 年训练缺口。Option Quotes 日内文件延迟 15 分钟，EOD 文件隔夜或次晨交付；冻结 Stage1 的 09:29 决策只能考虑**此前交易日**的合格报价，并须证明它在该时刻前已经送达。

**决定：`needs_SOXX_specific_historical_quote_sample_and_2011_gap_resolution`。** 本卡没有读取 SOXX 收益、构造 25 Delta 恒定期限偏斜、拟合模型或运行账户。下一步先取得供应商对 SOXX 在 2012、2017、2022 年原始期权链的有权限样本和报价/交付时刻，并确定 2011 年只能保持未知还是有另一个合格来源；购买前还要核对报价完整度、拆分后的合约身份、IV/Delta 正负及缺失率。不得从 AAPL/SPX/SPY 通用样本推断 SOXX 可用，更不能声称超过 SOXX。

公开采购页的只读报价观察见 `live_vendor_quote_observation.json`：2012-01-04 单日 SOXX、含 IV/Greeks 的 Cboe EOD 与 390 分钟报价分别显示 **10 美元**小计，但页面价格不保证该日实际交付完整 SOXX 合约链。另一条 [ORATS Delayed Data API](https://orats.com/data-api) 当前显示 **199 美元/月、20,000 请求**和 2007 年起的一般历史；[ORATS 近收盘历史档案](https://orats.com/near-eod-data)为一次性 **599 美元**、约 **500 GB**、14 天下载窗口，99 美元/月的 FTP 服务仅含购买后新数据。已有公开单日 SOXX 样本，仍未证明 2011–2022 全期 SOXX 覆盖、原始送达时间及 NAS 归档许可。未加入购物车、联络供应商或购买。

已准备 [ORATS 询样草稿](orats_inquiry_draft.md)，六个抽样日期均在冻结 Stage1 Qlib 交易日历内。发送给供应商需要用户明确授权；截至本卡更新，询样仍未发送。

**2026-10-23 来源更正：** [Cboe 官方 FAQ](https://datashop.cboe.com/faqs)把 Option Quotes 和 Option EOD Summary 的总体起点列为2010年1月，但两个现售[Option Quotes](https://datashop.cboe.com/option-quote-intervals)、[EOD Summary](https://datashop.cboe.com/option-eod-summary)产品页写2012年1月。不能据产品页断言Cboe无2011数据，也不能据FAQ断言2011 SOXX及IV/Delta可交付；具体冲突与询样问题见 `cboe_start_conflict_addendum.json`。旧卡对“2011缺口未证明可填”的决定不变。

**现售界面核查：** Option Quotes 与 Option EOD Summary 的历史日期选择器均只列2012–2026年，购物车为0、未提交表单。这与两个产品页一致，但不能排除FAQ所称2010起的旧版/人工交付渠道。2011 SOXX 加Calcs的实际可交付性仍须供应商确认；见 `cboe_checkout_ui_preflight.json`。
